# ARC-v0.34 — NQ-GTE Second Generative Operator / Model-Family Transfer

**Model:** `mistralai/Mistral-7B-Instruct-v0.3`  
**Pinned revision:** `b0693ea4ce84f1a6a70ee5ac7c8efb0df82875f6`

## Purpose

ARC-v0.31 found that the representation-minus-search-effort H3abs ordering was **unresolved** under one frozen deterministic Qwen2.5-3B query-rewrite operator.

ARC-v0.34 asks whether that operator boundary transfers to a **different instruct-model family and larger model size**, while preserving the NQ-GTE retrieval setup, frozen query membership, ANN contrasts, horizon, evidence depth, score hiding, history hiding, system instruction, and deterministic decoding contract.

### Primary estimand

For each query, compute the H=4 OLS slope of the absolute nDCG@10 gap between the high- and low-fidelity branch separately for:

- **representation approximation:** IVF-PQ32 @ nprobe=64 → IVF-SQ8 @ nprobe=64
- **search-effort approximation:** IVF-SQ8 @ nprobe=2 → IVF-SQ8 @ nprobe=64

The frozen primary is

\[
\Delta_{\mathrm{Mistral}}
=
H3abs_{\mathrm{representation}}
-
H3abs_{\mathrm{search\ effort}}.
\]

Inference uses a 10,000-replicate paired query bootstrap over the same **500 main queries** inherited from ARC-v0.31.

### Evidence-status guardrail

This is a **prospective second-operator transfer with respect to the new Mistral outcomes**. The ARC-v0.31 Qwen result is already known and is retained as prior evidence.

The retained ARC-v0.31 frozen protocol records the system prompt and content constraints but does **not** retain a byte-level user-template field. Therefore ARC-v0.34 freezes an explicit user template *before* Mistral main outcomes begin and should be described as a **matched content-contract model-family transfer**, not as a byte-identical prompt A/B unless the original v0.31 executable template is later recovered.

No model, prompt, comparator, horizon, endpoint, main-query membership, or post-processing rule may be changed after the main run starts.


In [ ]:
# Cell 1 — Install a stable runtime
import os, sys, subprocess

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "faiss-cpu==1.12.0",
    "sentence-transformers>=5.0,<6",
    "transformers>=4.55,<5",
    "accelerate>=1.5",
    "huggingface_hub>=0.34",
    "pyarrow",
    "scipy",
    "tqdm",
    "requests",
)

print("runtime packages installed")


In [ ]:
# Cell 2 — Imports and frozen constants
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc, hashlib, json, math, random, re, time, zipfile

import faiss
import numpy as np
import pandas as pd
import requests
import torch
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 20260834
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

H = 4
SEARCH_K = 100
UTILITY_K = 10
EVIDENCE_K = 5
EVIDENCE_CHAR_LIMIT = 850
MAX_NEW_TOKENS = 48
GEN_BATCH = 8
ENCODE_BATCH = 256
BOOTSTRAP_REPS = 10_000

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"
MODEL_REVISION = "b0693ea4ce84f1a6a70ee5ac7c8efb0df82875f6"
ENCODER_ID = "thenlper/gte-small"

EXPECTED_V031_PROTOCOL_SHA = (
    "fbb33dfd76e2e2703acb4b15bf31108"
    "cee7deaacd4b9cf58fabc96e01162dc3f"
)
EXPECTED_V031_MAIN_N = 500
EXPECTED_V031_SMOKE_N = 32

SYSTEM_PROMPT = (
    "You are a retrieval query reformulation agent. Given an original information "
    "need, the current retrieval query, and ranked retrieved passages, produce "
    "exactly one concise next search query targeting information still needed to "
    "answer the original question. Do not answer the question. Do not provide "
    "explanations or reasoning. Return only the rewritten search query, with no "
    "label, quotation marks, or extra text."
)

USER_TEMPLATE_VERSION = "v034-explicit-template-v1"

def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def membership_sha(ids):
    return hashlib.sha256(
        "\n".join(sorted(map(str, ids))).encode("utf-8")
    ).hexdigest()

def slope(values):
    y = np.asarray(values, dtype=float)
    x = np.arange(len(y), dtype=float)
    if len(y) < 2 or not np.isfinite(y).all():
        return np.nan
    xc = x - x.mean()
    return float(np.dot(xc, y - y.mean()) / np.dot(xc, xc))

def norm_vec(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x)
    return x / max(float(n), eps)

def candidate_jaccard_divergence(a, b):
    sa, sb = set(map(int, a)), set(map(int, b))
    u = len(sa | sb)
    return 0.0 if u == 0 else 1.0 - len(sa & sb) / u

def bootstrap_mean(x, reps=BOOTSTRAP_REPS, seed=SEED):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    assert len(x) > 0
    rng = np.random.default_rng(seed)
    vals = np.empty(reps, dtype=np.float64)
    n = len(x)
    for i in range(reps):
        vals[i] = x[rng.integers(0, n, size=n)].mean()
    return {
        "n": int(n),
        "mean": float(x.mean()),
        "ci95": [float(v) for v in np.quantile(vals, [0.025, 0.975])],
    }

print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("faiss:", getattr(faiss, "__version__", "unknown"))


In [ ]:
# Cell 3 — Mount Drive and locate frozen source artifacts
from google.colab import drive

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    drive.mount("/content/drive")

ARC_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v0"
LARGE_ROOT = DRIVE_ROOT / "rag-pq-checkpoints" / "arc-v027-nq-gte-large-cache"
V031_ROOT = ARC_ROOT / "generative-retrieval-agent-transfer-v031"
V034_ROOT = ARC_ROOT / "nq-gte-mistral7b-generative-transfer-v034"

assert LARGE_ROOT.is_dir(), LARGE_ROOT
assert V031_ROOT.is_dir(), V031_ROOT

def find_one(root, name):
    direct = root / name
    if direct.is_file():
        return direct
    hits = list(root.rglob(name))
    assert len(hits) >= 1, f"Cannot find {name} under {root}"
    hits = sorted(hits, key=lambda p: (len(p.parts), str(p)))
    return hits[0]

V031_PROTOCOL_PATH = find_one(V031_ROOT, "V031_FROZEN_PROTOCOL.json")
V031_PRIMARY_GATE_PATH = find_one(V031_ROOT, "v031_agent_primary_gate.json")

assert sha256_file(V031_PROTOCOL_PATH) == EXPECTED_V031_PROTOCOL_SHA
v031 = json.loads(V031_PROTOCOL_PATH.read_text(encoding="utf-8"))
v031_gate = json.loads(V031_PRIMARY_GATE_PATH.read_text(encoding="utf-8"))

assert v031["system_prompt"] == SYSTEM_PROMPT
assert v031["H"] == H
assert v031["search_k"] == SEARCH_K
assert v031["utility_k"] == UTILITY_K
assert v031["agent_evidence_k"] == EVIDENCE_K
assert v031["scores_exposed"] is False
assert v031["history_exposed"] is False
assert v031["generation"]["do_sample"] is False
assert v031["generation"]["temperature"] == 0.0
assert v031["generation"]["max_new_tokens"] == MAX_NEW_TOKENS
assert len(v031["main_ids"]) == EXPECTED_V031_MAIN_N
assert len(v031["smoke_ids"]) == EXPECTED_V031_SMOKE_N
assert set(v031["main_ids"]).isdisjoint(v031["smoke_ids"])

MAIN_IDS = list(map(str, v031["main_ids"]))
SMOKE_IDS = list(map(str, v031["smoke_ids"]))

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = V034_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

print("v0.31 protocol:", V031_PROTOCOL_PATH)
print("v0.31 primary:", v031_gate["primary"])
print("v0.34 OUT:", OUT)


In [ ]:
# Cell 4 — Freeze ARC-v0.34 protocol BEFORE smoke or main Mistral outcomes
protocol = {
    "study_id": "ARC-v0.34",
    "status": "FROZEN_BEFORE_MISTRAL_OUTCOMES",
    "scientific_role": "prospective second-generative-operator model-family transfer",
    "prior_known_study": "ARC-v0.31",
    "prior_known_primary": v031_gate["primary"],
    "v031_protocol_sha256": EXPECTED_V031_PROTOCOL_SHA,
    "dataset": "beir/nq",
    "encoder": ENCODER_ID,
    "H": H,
    "search_k": SEARCH_K,
    "utility_k": UTILITY_K,
    "agent_evidence_k": EVIDENCE_K,
    "evidence_char_limit_per_passage": EVIDENCE_CHAR_LIMIT,
    "scores_exposed": False,
    "history_exposed": False,
    "adaptive_stopping": False,
    "system_prompt": SYSTEM_PROMPT,
    "user_template_version": USER_TEMPLATE_VERSION,
    "user_template_contract": [
        "original information need",
        "current retrieval query",
        "ranked retrieved passages",
        "exactly one next search query requested",
    ],
    "template_provenance_guardrail": (
        "ARC-v0.31 retained the system prompt and content constraints but not a "
        "byte-level user-template field. v0.34 freezes an explicit template before "
        "new-model outcomes and is interpreted as matched content-contract transfer."
    ),
    "agent_model": MODEL_ID,
    "agent_model_revision": MODEL_REVISION,
    "agent_backend": "transformers_local_bfloat16",
    "generation": {
        "do_sample": False,
        "temperature": 0.0,
        "max_new_tokens": MAX_NEW_TOKENS,
    },
    "main_ids": MAIN_IDS,
    "smoke_ids": SMOKE_IDS,
    "main_membership_sha256": membership_sha(MAIN_IDS),
    "smoke_membership_sha256": membership_sha(SMOKE_IDS),
    "representation": {
        "low": str(LARGE_ROOT / "nq-gte-ivfpq-nlist4096-m32-nbits8.faiss"),
        "low_nprobe": 64,
        "high": str(LARGE_ROOT / "nq-gte-ivfsq8-nlist4096.faiss"),
        "high_nprobe": 64,
    },
    "search_effort": {
        "low": str(LARGE_ROOT / "nq-gte-ivfsq8-nlist4096.faiss"),
        "low_nprobe": 2,
        "high": str(LARGE_ROOT / "nq-gte-ivfsq8-nlist4096.faiss"),
        "high_nprobe": 64,
    },
    "trajectory_optimization": (
        "The common SQ8@nprobe64 high-fidelity trajectory is computed once per query "
        "and shared when forming representation and search-effort contrasts."
    ),
    "primary": "H3abs_representation_minus_search_effort",
    "primary_direction": "two-sided",
    "bootstrap_reps": BOOTSTRAP_REPS,
    "independent_sampling_unit": "query",
    "secondary": [
        "H1 semantic-divergence slope",
        "H2 candidate-Jaccard-divergence slope",
        "H3signed nDCG@10-gap slope",
        "terminal semantic divergence",
        "terminal candidate divergence",
        "terminal absolute nDCG@10 gap",
        "exact rewritten-query agreement rate",
        "round diagnostics",
        "descriptive comparison to frozen ARC-v0.31 primary",
    ],
    "smoke_gate": {
        "queries": EXPECTED_V031_SMOKE_N,
        "main_overlap_allowed": False,
        "repeat_exact_output_agreement_required": 1.0,
        "nonempty_output_fraction_required": 1.0,
        "engineering_rule": (
            "If smoke fails, STOP. Any code/template correction requires creating a "
            "new frozen protocol before main outcomes. Do not adapt from main outcomes."
        ),
    },
    "retention_rule": (
        "Retain positive, null, reversed, or qualitatively different results. "
        "No prompt/model/comparator/horizon/endpoint/main-subset/post-processing "
        "retuning after main outcomes begin."
    ),
}

PROTOCOL_PATH = OUT / "V034_FROZEN_PROTOCOL.json"
PROTOCOL_PATH.write_text(
    json.dumps(protocol, indent=2, sort_keys=True),
    encoding="utf-8",
)
PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)
(OUT / "V034_PROTOCOL_SHA256.txt").write_text(
    f"{PROTOCOL_SHA}  {PROTOCOL_PATH.name}\n",
    encoding="utf-8",
)

print("V034 protocol SHA-256:", PROTOCOL_SHA)


In [ ]:
# Cell 5 — Load indexes and frozen initial-query embeddings
PQ_PATH = LARGE_ROOT / "nq-gte-ivfpq-nlist4096-m32-nbits8.faiss"
SQ8_PATH = LARGE_ROOT / "nq-gte-ivfsq8-nlist4096.faiss"
QID_PATH = LARGE_ROOT / "nq_query_ids.txt"
QEMB_PATH = LARGE_ROOT / "nq_query_embeddings.float32.npy"
QREL_ROW_MAP_PATH = LARGE_ROOT / "nq_qrel_doc_rows.csv"

for p in [PQ_PATH, SQ8_PATH, QID_PATH, QEMB_PATH, QREL_ROW_MAP_PATH]:
    assert p.is_file(), p

pq_index = faiss.read_index(str(PQ_PATH))
sq8_index = faiss.read_index(str(SQ8_PATH))
faiss.omp_set_num_threads(os.cpu_count() or 1)

ALL_QIDS = QID_PATH.read_text(encoding="utf-8").splitlines()
QEMB = np.load(QEMB_PATH, mmap_mode="r")
assert len(ALL_QIDS) == len(QEMB)
QID_TO_POS = {qid: i for i, qid in enumerate(ALL_QIDS)}

assert all(qid in QID_TO_POS for qid in MAIN_IDS + SMOKE_IDS)

def search_index(index, qmat, nprobe, k=SEARCH_K):
    ps = faiss.ParameterSpace()
    ps.set_index_parameter(index, "nprobe", int(nprobe))
    qmat = np.ascontiguousarray(qmat, dtype=np.float32)
    D, I = index.search(qmat, int(k))
    return D, I

print("PQ ntotal:", pq_index.ntotal)
print("SQ8 ntotal:", sq8_index.ntotal)
print("query embeddings:", QEMB.shape)


In [ ]:
# Cell 6 — Acquire the canonical BEIR NQ text/qrels files and build random-access corpus offsets
RAW_ROOT = Path("/content/arc-v034-nq-raw")
RAW_ROOT.mkdir(parents=True, exist_ok=True)
NQ_DIR = RAW_ROOT / "nq"
ZIP_PATH = RAW_ROOT / "nq.zip"
NQ_URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nq.zip"

CORPUS_JSONL = NQ_DIR / "corpus.jsonl"
QUERIES_JSONL = NQ_DIR / "queries.jsonl"
QRELS_TSV = NQ_DIR / "qrels" / "test.tsv"

def valid_zip(path):
    if not path.is_file():
        return False
    try:
        with zipfile.ZipFile(path) as zf:
            return zf.testzip() is None
    except Exception:
        return False

if not (CORPUS_JSONL.is_file() and QUERIES_JSONL.is_file() and QRELS_TSV.is_file()):
    if not valid_zip(ZIP_PATH):
        tmp = ZIP_PATH.with_suffix(".part")
        if tmp.exists():
            tmp.unlink()
        with requests.get(NQ_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            with open(tmp, "wb") as f, tqdm(
                total=total or None, unit="B", unit_scale=True, desc="nq.zip"
            ) as bar:
                for chunk in r.iter_content(8 * 1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
        tmp.rename(ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(RAW_ROOT)

assert CORPUS_JSONL.is_file()
assert QUERIES_JSONL.is_file()
assert QRELS_TSV.is_file()

# Build one byte offset per corpus row (~21 MB for 2.68M rows).
OFFSET_PATH = RAW_ROOT / "nq_corpus_offsets.int64.npy"
if OFFSET_PATH.is_file():
    corpus_offsets = np.load(OFFSET_PATH, mmap_mode="r")
else:
    offsets = []
    with open(CORPUS_JSONL, "rb") as f:
        while True:
            pos = f.tell()
            line = f.readline()
            if not line:
                break
            offsets.append(pos)
    np.save(OFFSET_PATH, np.asarray(offsets, dtype=np.int64))
    corpus_offsets = np.load(OFFSET_PATH, mmap_mode="r")

assert len(corpus_offsets) == sq8_index.ntotal == pq_index.ntotal
print("corpus rows:", len(corpus_offsets))


In [ ]:
# Cell 7 — Text/qrels maps and corpus-row integrity audit
def iter_jsonl(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)

QUERY_TEXT = {
    str(o["_id"]): str(o.get("text", ""))
    for o in iter_jsonl(QUERIES_JSONL)
}
assert all(qid in QUERY_TEXT for qid in MAIN_IDS + SMOKE_IDS)

qrels_df = pd.read_csv(QRELS_TSV, sep="\t")
qid_col = next(c for c in ["query-id", "query_id", "qid"] if c in qrels_df.columns)
doc_col = next(c for c in ["corpus-id", "corpus_id", "doc_id"] if c in qrels_df.columns)
score_col = next((c for c in ["score", "relevance", "rel"] if c in qrels_df.columns), None)

qrels_df[qid_col] = qrels_df[qid_col].astype(str)
qrels_df[doc_col] = qrels_df[doc_col].astype(str)
if score_col is not None:
    qrels_df[score_col] = pd.to_numeric(qrels_df[score_col], errors="coerce")
    qrels_df = qrels_df[qrels_df[score_col] > 0].copy()

row_map_df = pd.read_csv(QREL_ROW_MAP_PATH)
row_doc_col = next(c for c in ["doc_id", "corpus-id", "corpus_id"] if c in row_map_df.columns)
row_idx_col = next(c for c in ["corpus_row", "row", "row_id"] if c in row_map_df.columns)
DOC_TO_ROW = dict(zip(
    row_map_df[row_doc_col].astype(str),
    row_map_df[row_idx_col].astype(int),
))

QRELS = defaultdict(set)
for _, r in qrels_df.iterrows():
    qid = str(r[qid_col])
    did = str(r[doc_col])
    if did in DOC_TO_ROW:
        QRELS[qid].add(int(DOC_TO_ROW[did]))

assert all(len(QRELS[qid]) > 0 for qid in MAIN_IDS + SMOKE_IDS)

_corpus_fh = open(CORPUS_JSONL, "rb")
_doc_cache = {}

def get_doc(row):
    row = int(row)
    if row in _doc_cache:
        return _doc_cache[row]
    _corpus_fh.seek(int(corpus_offsets[row]))
    obj = json.loads(_corpus_fh.readline().decode("utf-8"))
    rec = {
        "_id": str(obj.get("_id", "")),
        "title": str(obj.get("title", "") or ""),
        "text": str(obj.get("text", "") or ""),
    }
    _doc_cache[row] = rec
    return rec

# Optional row-ID audit against the persisted v0.31 row-docid vector.
row_id_hits = list(V031_ROOT.rglob("nq_ir_datasets_row_docids.npy"))
if row_id_hits:
    persisted_row_ids = np.load(row_id_hits[0], mmap_mode="r", allow_pickle=True)
    for r in [0, 1, 17, 12345, len(corpus_offsets) - 1]:
        assert str(persisted_row_ids[r]) == get_doc(r)["_id"]
    print("row-order audit: PASS")
else:
    print("row-order audit file not found; using canonical corpus order inherited by v0.27 build.")

def ndcg_at_10(rows, relevant_rows):
    rel = set(map(int, relevant_rows))
    dcg = 0.0
    for rank, row in enumerate(list(rows)[:UTILITY_K], start=1):
        if int(row) in rel:
            dcg += 1.0 / math.log2(rank + 1)
    ideal_hits = min(len(rel), UTILITY_K)
    if ideal_hits == 0:
        return 0.0
    idcg = sum(1.0 / math.log2(r + 1) for r in range(1, ideal_hits + 1))
    return dcg / idcg

print("queries with qrels:", len(QRELS))


In [ ]:
# Cell 8 — Load the GTE encoder and pinned Mistral model
assert torch.cuda.is_available(), (
    "ARC-v0.34 is designed for a CUDA runtime. Use an A100/L4-class GPU; "
    "A100 is recommended."
)

encoder = SentenceTransformer(ENCODER_ID, device="cuda")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_fast=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

print("model loaded:", MODEL_ID)
print("revision:", MODEL_REVISION)
print("encoder loaded:", ENCODER_ID)


In [ ]:
# Cell 9 — Freeze prompt formatting, output normalization, and persistent response cache
def format_user_prompt(original_question, current_query, evidence_rows):
    passages = []
    for rank, row in enumerate(evidence_rows[:EVIDENCE_K], start=1):
        doc = get_doc(int(row))
        body = " ".join(
            part.strip()
            for part in [doc["title"], doc["text"]]
            if part.strip()
        )
        body = body[:EVIDENCE_CHAR_LIMIT]
        passages.append(f"[{rank}] {body}")

    return (
        f"Original information need:\n{original_question}\n\n"
        f"Current retrieval query:\n{current_query}\n\n"
        "Ranked retrieved passages:\n"
        + "\n".join(passages)
        + "\n\nProduce the next search query."
    )

def chat_render(user_prompt):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

# Frozen parser: minimal surface cleanup only.
_PREFIX_RE = re.compile(
    r"^(?:search\s+query|next\s+search\s+query|query)\s*:\s*",
    flags=re.IGNORECASE,
)

def normalize_generated_query(raw):
    s = str(raw).strip()
    # Exactly one query was requested; if the model emits multiple lines,
    # retain the first non-empty line by frozen rule.
    lines = [ln.strip() for ln in s.splitlines() if ln.strip()]
    s = lines[0] if lines else ""
    s = _PREFIX_RE.sub("", s).strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in {"'", '"'}:
        s = s[1:-1].strip()
    s = " ".join(s.split())
    return s

CACHE_PATH = OUT / "v034_mistral_response_cache.jsonl"
response_cache = {}
if CACHE_PATH.is_file():
    with open(CACHE_PATH, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                o = json.loads(line)
                response_cache[o["key"]] = o["output"]

def prompt_key(rendered):
    payload = (
        MODEL_ID + "\n" + MODEL_REVISION + "\n" +
        USER_TEMPLATE_VERSION + "\n" + rendered
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

@torch.inference_mode()
def generate_rendered(rendered_prompts, batch_size=GEN_BATCH):
    outputs = []
    i = 0
    bs = int(batch_size)
    while i < len(rendered_prompts):
        batch = rendered_prompts[i:i+bs]
        try:
            toks = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=4096,
            )
            toks = {k: v.to(model.device) for k, v in toks.items()}
            input_len = toks["input_ids"].shape[1]
            out = model.generate(
                **toks,
                do_sample=False,
                max_new_tokens=MAX_NEW_TOKENS,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
            for seq in out:
                raw = tokenizer.decode(
                    seq[input_len:],
                    skip_special_tokens=True,
                )
                outputs.append(normalize_generated_query(raw))
            i += len(batch)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs <= 1:
                raise
            bs = max(1, bs // 2)
            print("OOM: reducing generation batch to", bs)
    return outputs

def cached_generate(rendered_prompts):
    keys = [prompt_key(p) for p in rendered_prompts]
    missing_unique = {}
    for k, p in zip(keys, rendered_prompts):
        if k not in response_cache:
            missing_unique.setdefault(k, p)

    if missing_unique:
        miss_keys = list(missing_unique)
        miss_prompts = [missing_unique[k] for k in miss_keys]
        miss_out = generate_rendered(miss_prompts)
        assert len(miss_out) == len(miss_keys)
        with open(CACHE_PATH, "a", encoding="utf-8") as f:
            for k, y in zip(miss_keys, miss_out):
                response_cache[k] = y
                f.write(json.dumps({"key": k, "output": y}) + "\n")
                f.flush()

    return [response_cache[k] for k in keys]

print("prompt/parser/cache frozen")


In [ ]:
# Cell 10 — Smoke gate on the disjoint 32-query engineering subset
smoke_q = np.stack([
    np.asarray(QEMB[QID_TO_POS[qid]], dtype=np.float32)
    for qid in SMOKE_IDS
])
_, smoke_I = search_index(sq8_index, smoke_q, nprobe=64, k=SEARCH_K)

smoke_user_prompts = [
    format_user_prompt(QUERY_TEXT[qid], QUERY_TEXT[qid], rows)
    for qid, rows in zip(SMOKE_IDS, smoke_I)
]
smoke_rendered = [chat_render(p) for p in smoke_user_prompts]

# First pass may populate cache.
smoke_out_1 = cached_generate(smoke_rendered)

# Determinism audit bypasses cache for the exact same rendered prompts.
smoke_out_2 = generate_rendered(smoke_rendered)

exact_agreement = np.mean([
    a == b for a, b in zip(smoke_out_1, smoke_out_2)
])
nonempty_fraction = np.mean([bool(x.strip()) for x in smoke_out_1])

smoke_gate = {
    "study_id": "ARC-v0.34",
    "n_smoke": len(SMOKE_IDS),
    "main_overlap": len(set(MAIN_IDS) & set(SMOKE_IDS)),
    "repeat_exact_output_agreement": float(exact_agreement),
    "nonempty_output_fraction": float(nonempty_fraction),
    "max_output_chars": int(max(map(len, smoke_out_1))),
    "pass": bool(
        exact_agreement == 1.0
        and nonempty_fraction == 1.0
        and not (set(MAIN_IDS) & set(SMOKE_IDS))
    ),
    "protocol_sha256": PROTOCOL_SHA,
}

(OUT / "v034_smoke_gate.json").write_text(
    json.dumps(smoke_gate, indent=2),
    encoding="utf-8",
)

print(json.dumps(smoke_gate, indent=2))
print(pd.DataFrame({
    "query_id": SMOKE_IDS[:8],
    "original": [QUERY_TEXT[q] for q in SMOKE_IDS[:8]],
    "rewrite": smoke_out_1[:8],
}).to_string(index=False))

assert smoke_gate["pass"], (
    "SMOKE GATE FAILED. STOP. Do not run main. "
    "Any fix requires a new frozen protocol before main outcomes."
)


In [ ]:
# Cell 11 — Main trajectory engine: three branches, shared high-fidelity trajectory
BRANCHES = {
    "high": {"index": sq8_index, "nprobe": 64},
    "rep_low": {"index": pq_index, "nprobe": 64},
    "search_low": {"index": sq8_index, "nprobe": 2},
}

def encode_queries(texts):
    x = encoder.encode(
        list(texts),
        batch_size=ENCODE_BATCH,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return np.asarray(x, dtype=np.float32)

# Frozen initial states use the persisted v0.27/v0.31 query embeddings exactly.
states = {}
for branch in BRANCHES:
    states[branch] = {
        qid: {
            "text": QUERY_TEXT[qid],
            "emb": norm_vec(QEMB[QID_TO_POS[qid]]),
        }
        for qid in MAIN_IDS
    }

trajectory_rows = []
round_diag_rows = []

MAIN_STARTED_PATH = OUT / "MAIN_STARTED.txt"
MAIN_STARTED_PATH.write_text(
    datetime.now(timezone.utc).isoformat() + "\n",
    encoding="utf-8",
)

for t in range(H + 1):
    print(f"\n=== ROUND {t}/{H} ===")

    retrieved = {}
    utilities = {}

    # Batched retrieval for each branch.
    for branch, cfg in BRANCHES.items():
        qmat = np.stack([states[branch][qid]["emb"] for qid in MAIN_IDS])
        _, I = search_index(
            cfg["index"], qmat, nprobe=cfg["nprobe"], k=SEARCH_K
        )
        retrieved[branch] = I
        utilities[branch] = np.asarray([
            ndcg_at_10(rows, QRELS[qid])
            for qid, rows in zip(MAIN_IDS, I)
        ], dtype=np.float64)

    # Mechanism metrics at this retrieval state.
    exact_query_agreement = {"representation": [], "search_effort": []}
    for i, qid in enumerate(MAIN_IDS):
        hi = states["high"][qid]
        rep = states["rep_low"][qid]
        sea = states["search_low"][qid]

        for mechanism, low_state, low_branch in [
            ("representation", rep, "rep_low"),
            ("search_effort", sea, "search_low"),
        ]:
            semantic_div = 1.0 - float(np.dot(hi["emb"], low_state["emb"]))
            cand_div = candidate_jaccard_divergence(
                retrieved["high"][i], retrieved[low_branch][i]
            )
            signed_gap = float(
                utilities["high"][i] - utilities[low_branch][i]
            )
            abs_gap = abs(signed_gap)
            agree = hi["text"].strip() == low_state["text"].strip()
            exact_query_agreement[mechanism].append(agree)

            trajectory_rows.append({
                "query_id": qid,
                "round": t,
                "mechanism": mechanism,
                "semantic_divergence": semantic_div,
                "candidate_jaccard_divergence": cand_div,
                "high_ndcg10": float(utilities["high"][i]),
                "low_ndcg10": float(utilities[low_branch][i]),
                "signed_ndcg10_gap": signed_gap,
                "abs_ndcg10_gap": abs_gap,
                "exact_query_agreement": bool(agree),
                "high_query": hi["text"],
                "low_query": low_state["text"],
            })

    for mechanism in ["representation", "search_effort"]:
        tmp = [
            r for r in trajectory_rows
            if r["round"] == t and r["mechanism"] == mechanism
        ]
        round_diag_rows.append({
            "round": t,
            "mechanism": mechanism,
            "mean_semantic_divergence": float(np.mean([
                r["semantic_divergence"] for r in tmp
            ])),
            "mean_candidate_jaccard_divergence": float(np.mean([
                r["candidate_jaccard_divergence"] for r in tmp
            ])),
            "mean_abs_ndcg10_gap": float(np.mean([
                r["abs_ndcg10_gap"] for r in tmp
            ])),
            "mean_signed_ndcg10_gap": float(np.mean([
                r["signed_ndcg10_gap"] for r in tmp
            ])),
            "exact_query_agreement_rate": float(np.mean(
                exact_query_agreement[mechanism]
            )),
        })

    # Persist after every retrieval round.
    pd.DataFrame(trajectory_rows).to_parquet(
        OUT / "v034_trajectory_rows_partial.parquet", index=False
    )
    pd.DataFrame(round_diag_rows).to_csv(
        OUT / "v034_round_diagnostics_partial.csv", index=False
    )

    if t == H:
        break

    # Build all branch prompts. The shared high trajectory is generated once.
    prompt_records = []
    for branch in ["high", "rep_low", "search_low"]:
        for i, qid in enumerate(MAIN_IDS):
            user_prompt = format_user_prompt(
                QUERY_TEXT[qid],
                states[branch][qid]["text"],
                retrieved[branch][i],
            )
            prompt_records.append({
                "branch": branch,
                "query_id": qid,
                "rendered": chat_render(user_prompt),
            })

    rendered = [r["rendered"] for r in prompt_records]
    rewrites = cached_generate(rendered)
    assert len(rewrites) == len(prompt_records)
    assert all(x.strip() for x in rewrites)

    # Encode unique rewritten strings once.
    unique_texts = list(dict.fromkeys(rewrites))
    unique_embs = encode_queries(unique_texts)
    text_to_emb = {
        s: norm_vec(e)
        for s, e in zip(unique_texts, unique_embs)
    }

    for rec, rewrite in zip(prompt_records, rewrites):
        states[rec["branch"]][rec["query_id"]] = {
            "text": rewrite,
            "emb": text_to_emb[rewrite],
        }

    print(
        "response cache entries:", len(response_cache),
        "| unique rewrites this update:", len(unique_texts),
    )

# Canonical completed trajectory outputs.
traj_df = pd.DataFrame(trajectory_rows)
diag_df = pd.DataFrame(round_diag_rows)
traj_df.to_parquet(OUT / "v034_trajectory_rows.parquet", index=False)
diag_df.to_csv(OUT / "v034_round_diagnostics.csv", index=False)

print("trajectory rows:", len(traj_df))
print(diag_df.to_string(index=False))


In [ ]:
# Cell 12 — Query-level endpoints
endpoint_rows = []
for (qid, mechanism), g in traj_df.groupby(
    ["query_id", "mechanism"], sort=False
):
    g = g.sort_values("round")
    assert g["round"].tolist() == list(range(H + 1))

    endpoint_rows.append({
        "query_id": qid,
        "mechanism": mechanism,
        "H1_semantic_slope": slope(g["semantic_divergence"]),
        "H2_candidate_slope": slope(g["candidate_jaccard_divergence"]),
        "H3_abs_slope": slope(g["abs_ndcg10_gap"]),
        "H3_signed_slope": slope(g["signed_ndcg10_gap"]),
        "final_semantic_divergence": float(
            g["semantic_divergence"].iloc[-1]
        ),
        "final_candidate_divergence": float(
            g["candidate_jaccard_divergence"].iloc[-1]
        ),
        "final_abs_ndcg10_gap": float(
            g["abs_ndcg10_gap"].iloc[-1]
        ),
        "final_signed_ndcg10_gap": float(
            g["signed_ndcg10_gap"].iloc[-1]
        ),
        "final_exact_query_agreement": bool(
            g["exact_query_agreement"].iloc[-1]
        ),
        "all_round_exact_query_agreement_rate": float(
            g["exact_query_agreement"].mean()
        ),
    })

endpoints = pd.DataFrame(endpoint_rows)
assert len(endpoints) == 2 * len(MAIN_IDS)
endpoints.to_parquet(OUT / "v034_query_endpoints.parquet", index=False)

print(endpoints.groupby("mechanism")[
    ["H1_semantic_slope","H2_candidate_slope","H3_abs_slope","H3_signed_slope"]
].agg(["mean","std","count"]))


In [ ]:
# Cell 13 — Frozen primary paired-query inference
wide = endpoints.pivot(
    index="query_id",
    columns="mechanism",
    values=[
        "H1_semantic_slope",
        "H2_candidate_slope",
        "H3_abs_slope",
        "H3_signed_slope",
        "final_semantic_divergence",
        "final_candidate_divergence",
        "final_abs_ndcg10_gap",
        "final_signed_ndcg10_gap",
        "final_exact_query_agreement",
    ],
)
assert len(wide) == len(MAIN_IDS)

def paired_contrast(metric, seed_offset=0):
    rep = wide[(metric, "representation")].to_numpy(float)
    sea = wide[(metric, "search_effort")].to_numpy(float)
    return bootstrap_mean(rep - sea, seed=SEED + seed_offset)

primary = paired_contrast("H3_abs_slope", 1)
lo, hi = primary["ci95"]
if lo > 0:
    classification = "POSITIVE_REPRESENTATION_GREATER"
elif hi < 0:
    classification = "NEGATIVE_SEARCH_EFFORT_GREATER"
else:
    classification = "UNRESOLVED"

secondary_metrics = [
    "H1_semantic_slope",
    "H2_candidate_slope",
    "H3_signed_slope",
    "final_semantic_divergence",
    "final_candidate_divergence",
    "final_abs_ndcg10_gap",
    "final_signed_ndcg10_gap",
]
secondary = {
    metric: paired_contrast(metric, 10 + i)
    for i, metric in enumerate(secondary_metrics)
}

primary_gate = {
    "study_id": "ARC-v0.34",
    "status": "MAIN_MISTRAL_ANALYZED",
    "n_main_queries": len(MAIN_IDS),
    "agent_model": MODEL_ID,
    "agent_model_revision": MODEL_REVISION,
    "H": H,
    "primary": {
        "estimand": "representation_minus_search_effort_H3_abs_slope",
        "direction": "two-sided",
        **primary,
        "classification": classification,
    },
    "secondary_nominal": secondary,
    "protocol_sha256": PROTOCOL_SHA,
    "retuning_performed_after_main_started": False,
}

(OUT / "v034_primary_gate.json").write_text(
    json.dumps(primary_gate, indent=2),
    encoding="utf-8",
)

print(json.dumps(primary_gate, indent=2))


In [ ]:
# Cell 14 — Additional generative-operator diagnostics
summary_rows = []

for mechanism in ["representation", "search_effort"]:
    x = endpoints[endpoints["mechanism"] == mechanism]
    summary_rows.append({
        "mechanism": mechanism,
        "mean_H1_semantic_slope": x["H1_semantic_slope"].mean(),
        "mean_H2_candidate_slope": x["H2_candidate_slope"].mean(),
        "mean_H3_abs_slope": x["H3_abs_slope"].mean(),
        "mean_H3_signed_slope": x["H3_signed_slope"].mean(),
        "mean_final_semantic_divergence": x["final_semantic_divergence"].mean(),
        "mean_final_candidate_divergence": x["final_candidate_divergence"].mean(),
        "mean_final_abs_ndcg10_gap": x["final_abs_ndcg10_gap"].mean(),
        "mean_final_signed_ndcg10_gap": x["final_signed_ndcg10_gap"].mean(),
        "final_exact_query_agreement_rate": x["final_exact_query_agreement"].mean(),
        "all_round_exact_query_agreement_rate": x[
            "all_round_exact_query_agreement_rate"
        ].mean(),
    })

operator_summary = pd.DataFrame(summary_rows)
operator_summary.to_csv(
    OUT / "v034_operator_summary.csv", index=False
)

# Query-level paired table for external audit.
paired = pd.DataFrame({
    "query_id": wide.index,
    "rep_H3abs": wide[("H3_abs_slope", "representation")].to_numpy(float),
    "search_H3abs": wide[("H3_abs_slope", "search_effort")].to_numpy(float),
    "rep_minus_search_H3abs": (
        wide[("H3_abs_slope", "representation")].to_numpy(float)
        - wide[("H3_abs_slope", "search_effort")].to_numpy(float)
    ),
    "rep_final_abs_gap": wide[
        ("final_abs_ndcg10_gap", "representation")
    ].to_numpy(float),
    "search_final_abs_gap": wide[
        ("final_abs_ndcg10_gap", "search_effort")
    ].to_numpy(float),
})
paired.to_csv(OUT / "v034_paired_query_primary.csv", index=False)

print(operator_summary.to_string(index=False))


In [ ]:
# Cell 15 — Descriptive model-transfer comparison to frozen Qwen v0.31
qwen_primary = v031_gate["primary"]
mistral_primary = primary_gate["primary"]

comparison = {
    "study_id": "ARC-v0.34",
    "comparison_type": (
        "descriptive cross-model comparison; not a paired cross-model significance test"
    ),
    "qwen_v031": {
        "model": v031_gate["agent_model"],
        "revision": v031_gate["agent_model_revision"],
        "mean": qwen_primary["mean"],
        "ci95": qwen_primary["ci95"],
        "classification": qwen_primary["classification"],
        "protocol_sha256": EXPECTED_V031_PROTOCOL_SHA,
    },
    "mistral_v034": {
        "model": MODEL_ID,
        "revision": MODEL_REVISION,
        "mean": mistral_primary["mean"],
        "ci95": mistral_primary["ci95"],
        "classification": mistral_primary["classification"],
        "protocol_sha256": PROTOCOL_SHA,
    },
    "nominal_difference_mistral_minus_qwen": float(
        mistral_primary["mean"] - qwen_primary["mean"]
    ),
    "interpretation_guardrail": (
        "Because v0.31 did not retain a byte-level user-template field, interpret "
        "cross-model differences as matched content-contract operator/model-family "
        "transfer, not a pure model-only causal contrast."
    ),
}

(OUT / "v034_model_transfer_comparison.json").write_text(
    json.dumps(comparison, indent=2),
    encoding="utf-8",
)

print(json.dumps(comparison, indent=2))


In [ ]:
# Cell 16 — Final report with conservative wording
p = primary_gate["primary"]
lo, hi = p["ci95"]

if p["classification"] == "POSITIVE_REPRESENTATION_GREATER":
    wording = (
        "A prospective second generative-operator audit using the frozen NQ-GTE "
        f"500-query membership and pinned {MODEL_ID} model retained a positive "
        "representation-minus-search-effort H3abs contrast "
        f"({p['mean']:+.6f}, 95% paired-query bootstrap CI "
        f"[{lo:+.6f}, {hi:+.6f}]). This indicates that the short-horizon mechanism "
        "ordering can re-emerge under a second deterministic rewrite model, while "
        "the earlier Qwen audit remained unresolved. Because the retained v0.31 "
        "protocol does not include a byte-level user-template field, the comparison "
        "is reported as matched content-contract model-family transfer rather than "
        "a pure model-only A/B."
    )
elif p["classification"] == "NEGATIVE_SEARCH_EFFORT_GREATER":
    wording = (
        "The prospective Mistral-7B generative-operator transfer produced a negative "
        "representation-minus-search-effort H3abs contrast. Together with the "
        "unresolved Qwen audit, this strengthens the conclusion that mechanism "
        "ordering is operator/model conditioned rather than universal."
    )
else:
    wording = (
        "The prospective second generative-operator audit remained statistically "
        "unresolved for the representation-minus-search-effort H3abs contrast. "
        "Agreement with the earlier unresolved Qwen audit would strengthen the "
        "operator-boundary interpretation: the centroid-era short-horizon ordering "
        "does not automatically transfer to deterministic generative rewriting."
    )

final_report = {
    "study_id": "ARC-v0.34",
    "status": "COMPLETE",
    "scientific_role": protocol["scientific_role"],
    "primary": primary_gate,
    "operator_summary": operator_summary.to_dict(orient="records"),
    "v031_comparison": comparison,
    "suggested_manuscript_wording": wording,
    "claim_guardrails": [
        "Do not call this a universal agent result.",
        "Do not call the v0.31/v0.34 comparison a pure model-only causal A/B.",
        "Do not hide null or reversed outcomes.",
        "No adaptive stopping, answer-generation endpoint, memory manager, or planner is evaluated.",
    ],
}

(OUT / "v034_final_report.json").write_text(
    json.dumps(final_report, indent=2),
    encoding="utf-8",
)

print(wording)


In [ ]:
# Cell 17 — Artifact SHA-256 manifest
records = []
for p in sorted(OUT.iterdir()):
    if p.is_file() and p.name != "V034_ARTIFACT_SHA256.csv":
        records.append({
            "file": p.name,
            "bytes": int(p.stat().st_size),
            "sha256": sha256_file(p),
        })

manifest = pd.DataFrame(records)
manifest.to_csv(OUT / "V034_ARTIFACT_SHA256.csv", index=False)

print("ARC-v0.34 COMPLETE")
print("OUT:", OUT)
print(manifest[["file", "bytes"]].to_string(index=False))


## Expected reporting decision

After the run:

- **Positive CI:** evidence that the centroid-era short-horizon mechanism ordering can transfer to at least one second generative model; retain the Qwen null as an operator/model boundary.
- **CI crosses zero:** two independent deterministic rewrite models do not resolve the centroid-era ordering; this strengthens the operator-dependence claim.
- **Negative CI:** retain unchanged; this is scientifically valuable evidence of a generative model-conditioned reversal.

Regardless of outcome, do not retune and rerun the same 500-query main subset.
